# Zero-shot extraction with aibackends + GLiNER2.5 (CPU/GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/AIBackends-GLiNER2_5-extraction.ipynb)

This notebook demos the GLiNER2.5 extraction backend on **aibackends v0.7.0**
(introduced in v0.6.0, unchanged since).

[GLiNER2.5](https://huggingface.co/fastino/gliner2.5-base-v1) is a zero-shot extraction
model: you describe *what* to pull out of a text at call time — entity types, classification
tasks, relations — and it finds them **without any fine-tuning**. It runs **entirely on your
machine** — no API keys, no data leaving the runtime.

One model, three task families, all behind the same `aibackends` task API:

1. **Entity extraction** — span-level NER for any labels you name, plus per-span attributes
2. **Text classification** — single-label and multi-label tasks, with logical constraints
3. **Knowledge-graph extraction** — entities *and* relations in one pass

**What this notebook covers**

1. Setup and model variants (`small` / `base` / `multi`)
2. Load once, reuse everywhere
3. Zero-shot entity extraction
4. Structured entity attributes
5. Batching, thresholds, and long documents
6. Text classification — single, multi-label, and constrained
7. Knowledge-graph extraction (entities + relations)
8. Async calls
9. Doing the same from the CLI

> Runs fine on a **free CPU runtime**. A GPU runtime works too and is picked up automatically.

## Setup

The `extraction` extra pulls in `gliner2[local]` and `protobuf`.

In [ ]:
!pip install -q "aibackends[extraction]>=0.7.0"

# If a later import fails with a protobuf or transformers error, use
# Runtime > Restart session, then continue from the next cell.

In [ ]:
import time

import aibackends
from aibackends.backends.extraction import get_extraction_backend, list_extraction_backends

print('aibackends', aibackends.__version__)
print('extraction backends:', list_extraction_backends())

# GLiNER2.5 accepts: cpu, gpu, cuda, cuda:<index>, mps
try:
    import torch
    DEVICE = 'gpu' if torch.cuda.is_available() else 'cpu'
except ImportError:
    DEVICE = 'cpu'

print('device:', DEVICE)

## 1. Model variants

`model` accepts a variant shortcut or any Hugging Face repo id:

| Variant | Model | Good for |
|---|---|---|
| `small` | `fastino/gliner2.5-small-v1` | fastest, tightest latency budgets |
| `base` *(default)* | `fastino/gliner2.5-base-v1` | best speed/quality balance |
| `multi` | `fastino/gliner2.5-multi-v1` | multilingual text |

Everything below works on any variant — swap `model='small'` in to trade accuracy for speed.

## 2. Load once, reuse everywhere

The model is cached per process **and** per device, so you pay the load cost once.
The first run of the cell below also downloads ~800MB of weights from the Hugging Face Hub.

In [ ]:
backend = get_extraction_backend('gliner2.5')

t = time.perf_counter()
backend.load(model='base', device=DEVICE)
load_s = time.perf_counter() - t
print(f'model load: {load_s:.1f}s')

## 3. Zero-shot entity extraction

`extract_entities` takes any list of labels and returns typed, sorted spans with character
offsets and confidences. No training data, no fine-tuning — just name the types you want.

In [ ]:
from aibackends.tasks import extract_entities

email = (
    "Hi Sarah, this is David Chen from Northwind Analytics. "
    "Following up on our call last Tuesday about the $48,000 renewal — "
    "please e-sign the contract at https://northwind.example.com/renew "
    "or reach me at david.chen@northwind.example.com before March 3rd."
)

result = extract_entities(
    email,
    labels=['person', 'organization', 'money', 'date', 'email', 'url'],
    device=DEVICE,
)

print(result.model_dump_json(indent=2))

The result is a pydantic model with helpers — `by_label` filters spans by type:

In [ ]:
for entity in result.by_label('person') + result.by_label('organization'):
    print(f'{entity.label:14} {entity.text!r:22} chars {entity.start}-{entity.end}  p={entity.confidence:.2f}')

Because spans carry character offsets, you can highlight them in the original text:

In [ ]:
def highlight(result) -> str:
    marked = result.text
    for entity in sorted(result.entities, key=lambda e: e.start or 0, reverse=True):
        marked = marked[:entity.start] + f'[{entity.label.upper()}] ' + marked[entity.start:]
        marked = marked[:entity.end + 1] + '[/]' + marked[entity.end + 1:]
    return marked

print(highlight(result))

### Labels with descriptions

Labels can also be a `{type: description}` map — the description steers the model when a
type name alone is ambiguous.

In [ ]:
result = extract_entities(
    "Flight UA 832 departs SFO at 6:45 PM and lands at JFK around 11 PM.",
    labels={
        'flight_number': 'an airline flight designator like AA 1234',
        'airport': 'an airport identified by name or IATA code',
    },
    device=DEVICE,
)

for entity in result.entities:
    print(f'{entity.label:14} {entity.text!r}')

## 4. Structured entity attributes

GLiNER2.5 can also classify each extracted span with its own attribute groups. Define a
group with `labels`, optionally restrict it to certain entity types with `applies_to`, and
every returned span carries the decoded attribute.

Here each `product` span gets a `sentiment` attribute — so you learn not just *which*
products came up, but *how they were received*.

In [ ]:
from aibackends.tasks import extract_entities_batch

reviews = [
    "The AeroPress brews fast but the grinder is loud and overpriced.",
    "Battery life on the new watch is phenomenal; the strap feels cheap though.",
]

result = extract_entities(
    reviews[0],
    labels=['product'],
    attributes={
        'sentiment': {
            'labels': ['positive', 'negative', 'neutral'],
            'applies_to': ['product'],
        },
    },
    device=DEVICE,
)

for entity in result.entities:
    sentiment = entity.attributes.get('sentiment')
    print(f'{entity.text!r:12} -> {sentiment.labels} {sentiment.confidences}')

## 5. Batch, thresholds, and long documents

`extract_entities_batch` uses GLiNER2.5's native batch inference — faster per item than
looping. `threshold` (default `0.5`) controls the confidence bar; raise it to keep only
high-precision spans.

For texts longer than the model context, pass `long_document=True` — the backend
chunk-and-merges automatically with `chunk_size` / `chunk_overlap` controls.

In [ ]:
import pandas as pd

from aibackends.tasks import extract_entities_batch

texts = [
    "Marie Curie won two Nobel Prizes and worked at the University of Paris.",
    "Ada Lovelace wrote the first algorithm for Babbage's Analytical Engine.",
    "Alan Turing cracked the Enigma cipher at Bletchley Park in 1941.",
]

t = time.perf_counter()
results = extract_entities_batch(texts, labels=['person', 'organization', 'date'], device=DEVICE)
batch_s = time.perf_counter() - t

pd.DataFrame(
    {
        'text': [t[:40] + '...' for t in texts],
        'entities': [', '.join(f'{e.text} ({e.label})' for e in r.entities) for r in results],
    }
)

## 6. Text classification

The same model does zero-shot classification. `classify_text` takes a `tasks` map —
each entry is a named task with its own labels:

- **single-label** (default): pick exactly one option
- **`multi_label: True`**: pick any number of options

You get labels, confidences, and full probability tables back in one pass.

In [ ]:
from aibackends.tasks import classify_text

ticket = (
    "I've been charged twice for my February subscription and nobody has "
    "responded to my ticket in five days. This is unacceptable."
)

result = classify_text(
    ticket,
    tasks={
        'category': {'labels': ['billing', 'bug', 'feature_request', 'account']},
        'urgency': {'labels': ['low', 'medium', 'high']},
        'emotions': {'labels': ['angry', 'frustrated', 'satisfied', 'confused'], 'multi_label': True},
    },
    device=DEVICE,
)

print(result.model_dump_json(indent=2))

Handy accessors: `value(task)` for single-label picks, `values(task)` for multi-label lists:

In [ ]:
print('category :', result.value('category'))
print('urgency  :', result.value('urgency'))
print('emotions :', result.values('emotions'))

### Constrained classification

`constraints` adds logical rules between tasks. Each rule is a dict with a `kind`
(`implies`, `excludes`, `iff`) and `when` / `then` `[task, label]` pairs.

Here: *if* the model picks urgency `high`, the emotion list **must** include `angry`
or... well, must include what we say — a rule that pairs `urgency: high` with
`emotions: angry`. The solver only returns assignments that stay feasible.

In [ ]:
result = classify_text(
    ticket,
    tasks={
        'category': {'labels': ['billing', 'bug', 'feature_request', 'account']},
        'urgency': {'labels': ['low', 'medium', 'high']},
        'emotions': {'labels': ['angry', 'frustrated', 'satisfied', 'confused'], 'multi_label': True},
    },
    constraints=[
        {'kind': 'implies', 'when': ['urgency', 'high'], 'then': ['emotions', 'angry']},
     ],
    device=DEVICE,
)

print('feasible  :', result.feasible)
print('constrained:', result.constrained)
print('urgency   :', result.value('urgency'), '-> emotions:', result.values('emotions'))

## 7. Knowledge-graph extraction

`extract_graph` pulls entities *and* relations in one joint pass. Relations are defined as
`name: head type: tail type` triples, and a beam-search decoder assembles a coherent graph
(`no_self_loops=True` by default).

This is the task GLiNER2.5 was built for — a full structured read of unstructured text,
zero-shot.

In [ ]:
from aibackends.tasks import extract_graph

news = (
    "OpenAI CEO Sam Altman announced a partnership with Microsoft last week. "
    "The deal, backed by Satya Nadella, will bring new models to Azure customers."
)

graph = extract_graph(
    news,
    entities=['person', 'organization', 'product'],
    relations=[
        {'name': 'works_at', 'head': 'person', 'tail': 'organization'},
        {'name': 'partner_of', 'head': 'organization', 'tail': 'organization'},
    ],
    device=DEVICE,
)

print(f'entities : {len(graph.entities)}')
for entity in graph.entities:
    print(f'  {entity.id:3} {entity.type:14} {entity.text!r}')

print(f'relations: {len(graph.relations)}')
for head, relation, tail in graph.triples():
    print(f'  {head!r} --[{relation}]--> {tail!r}')

`graph.triples()` gives you `(head text, relation, tail text)` — ready to load into
NetworkX, a graph database, or an LLM prompt as grounding context.

In [ ]:
# e.g. feed the triples to an LLM as facts, or persist them
import json

records = [
    {'head': h, 'relation': r, 'tail': t} for h, r, t in graph.triples()
]
print(json.dumps(records, indent=2))

## 8. Async

Every task has an `_async` twin — handy when extraction sits in an async web handler next
to your model call. Colab supports top-level `await`.

In [ ]:
import asyncio

from aibackends.tasks import extract_entities_batch_async

t = time.perf_counter()
async_results = await extract_entities_batch_async(
    texts, labels=['person', 'organization', 'date'], device=DEVICE
)
print(f'{len(async_results)} results in {time.perf_counter() - t:.2f}s without blocking the loop')

## 9. From the CLI

Same tasks, no Python. `aibackends task` prints JSON, so it pipes into `jq` nicely.

In [ ]:
!aibackends task extract-entities \
  --input "Tim Cook met Sundar Pichai in Cupertino to discuss App Store policy." \
  --labels person,organization,location \
  --device cpu

In [ ]:
!aibackends task classify-text \
  --input "The app crashes every time I open the camera." \
  --labels billing,bug,feature_request \
  --device cpu

In [ ]:
!aibackends task extract-graph \
  --input "Ada Lovelace worked with Charles Babbage on the Analytical Engine." \
  --entities person,machine \
  --relation 'collaborated_with:person:person' 'contributed_to:person:machine' \
  --device cpu

## Recap

```python
from aibackends.tasks import (
    extract_entities, extract_entities_batch,
    classify_text, classify_texts,
    extract_graph,
)

extract_entities(text, labels=['person', 'organization'])        # zero-shot NER
extract_entities(text, labels=['product'],                       # + per-span attributes
                 attributes={'sentiment': {'labels': ['positive', 'negative', 'neutral']}})
classify_text(text, tasks={'category': {'labels': [...]},         # single + multi-label
                           'tags': {'labels': [...], 'multi_label': True}})
classify_text(text, tasks=..., constraints=[...])                 # with logical constraints
extract_graph(text, entities=[...], relations=[{'name': 'works_at',
                                                'head': 'person', 'tail': 'organization'}])
# ...plus _async variants of everything
```

Things worth remembering:

- The model loads once per process and device — preload it if first-request latency matters.
- Labels are free-form strings, or `{type: description}` maps for tricky types.
- Batch calls beat looping when you have more than a couple of items.
- Use `long_document=True` for texts beyond the model context; tune with `chunk_size` / `chunk_overlap`.
- `threshold` trades recall for precision; constraints guarantee feasible classification outputs.
- Same API, any variant: `model='small'` / `'base'` / `'multi'`, or any HF repo id.

Want a different extraction model behind the same API? Register your own backend with
`register_extraction_backend` and pass `backend='your-name'`.

**Links**

- Release notes — https://github.com/donvito/aibackends/releases
- Repo — https://github.com/donvito/aibackends
- Model cards — https://huggingface.co/fastino/gliner2.5-base-v1, https://huggingface.co/fastino/gliner2.5-small-v1, https://huggingface.co/fastino/gliner2.5-multi-v1